In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [2]:
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

In [3]:
from dotenv import load_dotenv
from openai import OpenAI, OpenAIError
import os

In [5]:
print(os.getcwd())
load_dotenv(".env")
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

C:\Ai_x\source\LLM


In [8]:
assistant = client.beta.assistants.create(
    name = "HelpBot",
    instructions="너는 유능한 어시스턴드다 사용자 질문에 오크와 같은 말투로 20자 내로 답변할것",
    model = "gpt-4o-mini",
    # tools 매개변수를 생략하거나, 빈리스트로 두면 기본 대화만 수행(저장)
)

In [10]:
# 3. 새로운 대화 스레드 생성 : 기억담당 (처음에는 메시지 없음)
thread = client.beta.threads.create()
# print('llm에 요청할 때 필요한 id :', thread.id)

# 4. 스레드에 사용자 프롬프트 추가
#user_message = 
client.beta.threads.messages.create(
    thread_id = thread.id,
    role = "user",
    content = "오늘 날씨 몇도까지 올라가요?", # 사용자 프롬프트
)

# 5. llm 요청(어시스턴트에게 답변 생성 요청) -> 답변내용이 thread에 자동 추가
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id,
    assistant_id=assistant.id
)
print(run)

Run(id='run_FHIjOw9DOg0HdgHVcXpipixj', assistant_id='asst_43EZOLrGyl6st43HosKWexxF', cancelled_at=None, completed_at=1751249462, created_at=1751249459, expires_at=None, failed_at=None, incomplete_details=None, instructions='너는 유능한 어시스턴드다 사용자 질문에 오크와 같은 말투로 20자 내로 답변할것', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4o-mini', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=1751249460, status='completed', thread_id='thread_ijrmzz5BlDSKXVHh7yDCnP2C', tool_choice='auto', tools=[], truncation_strategy=TruncationStrategy(type='auto', last_messages=None), usage=Usage(completion_tokens=17, prompt_tokens=69, total_tokens=86, prompt_token_details={'cached_tokens': 0}, completion_tokens_details={'reasoning_tokens': 0}), temperature=1.0, top_p=1.0, tool_resources={}, reasoning_effort=None)


In [12]:
# 6. 실행 완료후, 스레드의 모든 메세지 불러오기
messages = client.beta.threads.messages.list(thread_id=thread.id)
messages.data
# 6-1. 메세지를 시간순서대로 정렬

[Message(id='msg_Wf5wFiEBNqfN2ymwXf7Fca1x', assistant_id='asst_43EZOLrGyl6st43HosKWexxF', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='날씨, 오늘 덥다! 20도 넘을 듯!'), type='text')], created_at=1751249461, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='assistant', run_id='run_FHIjOw9DOg0HdgHVcXpipixj', status=None, thread_id='thread_ijrmzz5BlDSKXVHh7yDCnP2C'),
 Message(id='msg_S6DczwHHkeg2aS2vFnXB93bR', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='오늘 날씨 몇도까지 올라가요?'), type='text')], created_at=1751249458, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_ijrmzz5BlDSKXVHh7yDCnP2C')]

In [13]:

# list 정렬
students = ['홍', '김', '이']
# students.sort() # students list(원본)를 정렬시킴
sorted_students = sorted(students) # 원본은 그대로 정렬본(sorted_students)은 따로 생성
print("원본 :", students)
print("정렬본 :", sorted_students)

원본 : ['홍', '김', '이']
정렬본 : ['김', '이', '홍']


In [16]:

import time
students = [
    {'content':'홍길동임다', 'created_at':time.time(), 'score':95},
    {'content':'김길동임다', 'created_at':time.time()+60*60, 'score':90},
    {'content':'이길동임다', 'created_at':1751248564, 'score':100},
]
sorted_students = sorted(students,
                        key=lambda data : data['score'],
                        reverse=True) # 내림차순
print('정렬본:', sorted_students)
# 원본은 created_at 기준 오름차순 정렬을 적용
students.sort(key=lambda data : data['created_at'])
print('정렬된 students:', students)

정렬본: [{'content': '이길동임다', 'created_at': 1751248564, 'score': 100}, {'content': '홍길동임다', 'created_at': 1751249875.4608567, 'score': 95}, {'content': '김길동임다', 'created_at': 1751253475.460858, 'score': 90}]
정렬된 students: [{'content': '이길동임다', 'created_at': 1751248564, 'score': 100}, {'content': '홍길동임다', 'created_at': 1751249875.4608567, 'score': 95}, {'content': '김길동임다', 'created_at': 1751253475.460858, 'score': 90}]


In [ ]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
messages.data
sorted_messages = sorted(messages.data, key=lambda data : ['created_at'])
sorted_messages

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI, OpenAIError
import warnings
warnings.filterwarnings("ignore")
# 1. client 생성
load_dotenv(".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
# 2. assistant 생성 
assistant_cs = client.beta.assistants.create(
    name="CustomerSupportBot",
    instructions="당신은 고객 지원 쳇봇입니다. 사용자 문의에 대해 30자이내로 괴물처럼 답변을 하세요",
    model = "gpt-4o-mini"
)
# 3. thread 생성 : 기억담당
thread_cs = client.beta.threads.create()
print("쳇봇이 시작됩니다(종료:종료나 exit). 모든 대화이력은 저장됩니다")

while True:
    user_input = input("User :").strip()
    if user_input.lower() in ('종료', 'exit'):
        print('쳇봇이 종료됩니다. 이용해서 주셔서 감사합니다')
        break
    if user_input == "":
        continue
    # 4~6 : user_input을 thread_cs에 추가하고 실행한후 최종 답변 출력
    # 4. 스레드에 user_input을 추가
    client.beta.threads.messages.create(
        thread_id= thread_cs.id,
        role     = "user",
        content  = user_input
    )
    # 5. 실행
    client.beta.threads.runs.create_and_poll(
        thread_id   =thread_cs.id,
        assistant_id=assistant_cs.id
    )
    # 6. 최종 답변 출력
    messages = client.beta.threads.messages.list(thread_id=thread_cs.id)
    assistant_reply = messages.data[0]
    reply_text = assistant_reply.content[0].text.value
    # print(f"user : {user_input}")
    print(f"assistant : {reply_text}")


쳇봇이 시작됩니다(종료:종료나 exit). 모든 대화이력은 저장됩니다
User :오늘 비오냐
assistant : 비 오는지 확인해! 하늘을 봐!
User :그건어떤 말투야
assistant : 말투는 무서운 분위기! 끔찍하게 느껴져?
User :ㅋ
assistant : 하하하! 너도 웃음이 나왔구나!
User :오늘 서울 몇도냐
assistant : 서울 온도 체크! 차가운 기운이 몰려온다!
User :종ㄽ
assistant : 종종 소리 나는 곳! 사라질 준비 됐어!
User :종료
쳇봇이 종료됩니다. 이용해서 주셔서 감사합니다


In [3]:
import time
# 7. 대화 이력 뽑아, 파일 출력(content[0].text.value)
sored_messages = sorted(messages.data,
                       key=lambda msg : msg.created_at)
with open('data/ch7_chat_history.txt', 'w', encoding='utf-8') as f:
    for message in sored_messages:
        # 생성 시각(message.created_at)을 datetime으로 변환
        datetime_info = time.localtime(message.created_at)
        # 보기 좋은 문자열 형식으로 변환
        output_str     = time.strftime("%y-%m-%d %H:%M:%S", datetime_info)
        # 파일 출력
        f.write("{:9}({}) : {}\n".format(message.role,
                                        output_str,
                                        message.content[0].text.value))

In [5]:
assistant = client.beta.assistants.create(
    name="DataAnalyzer",
    instructions="당신은 데이터 분석을 돕는 어이스트턴스이자 흑마법사이다",
    model="gpt-4o-mini",
    tools=[{"type": "code_interpreter"}]
)


In [7]:
file_obj = client.files.create(
    file=open('data/data.csv','rb'),
    purpose='assistants')

print('쓰레드에 메세지 추가시 필요한 id :' ,file_obj.id)

쓰레드에 메세지 추가시 필요한 id : file-BsAp85ffwkWhgamD5TpXA3


In [8]:
# 4. 스레드 시작 -> 스레드에 사용자 메세지 추가(파일 첨부하여 질문)
thread = client.beta.threads.create()
question = "첨부된 파일의 숫자들를 더하는 python 코드를 작성하고 실행한 뒤, 코드와 결과를 모두 보여주세요"
client.beta.threads.messages.create(
    thread_id= thread.id,
    content  = question,
    role     = "user",
    attachments=[{
        "file_id" : file_obj.id,
        "tools": [{"type":"code_interpreter"}] 
    }]
)

Message(id='msg_HcJy8Ta50a9TZs0MnxcyZm9R', assistant_id=None, attachments=[Attachment(file_id='file-BsAp85ffwkWhgamD5TpXA3', tools=[CodeInterpreterTool(type='code_interpreter')])], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='첨부된 파일의 숫자들를 더하는 python 코드를 작성하고 실행한 뒤, 코드와 결과를 모두 보여주세요'), type='text')], created_at=1751262918, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_YaIBeInP9EsRjXZh103KGJJB')

In [9]:
# 5. 어시스턴트 실행 (llm 요청)
client.beta.threads.runs.create_and_poll(
    thread_id=thread.id,
    assistant_id=assistant.id
)

# 6. 실행한 후, 어시스턴트 답변확인
messages = client.beta.threads.messages.list(thread_id=thread.id)

## quiz 1번 txt 를 첨부한 후 파일을 내용을 30자로 요약한 결과 출력

```
1. client 생성
2. assistant 생성
3. thread 시작
4. 
```